## Parte 0 —  Tres fallas que no fallan

**Nombres: Jessica Ballesteros, Darlyn Ludeña, Miguel Álvarez**

### 0.a — El solver que no ejecuta

**python parte0/a_llm_sin_ejecutar.py . Le da al LLM el enunciado de la Tarea A completo, sin herramientas, y
le pide el reporte; después corre el experimento de verdad —diez líneas de scikit-learn— y pone una cifra al lado
de la otra. Con la H200, el 2026-10-02, el reporte traía 30 cifras con decimales, todas con aspecto de medidas: la
exactitud del Naive Bayes en 0,9415 (medida: 0,9474), la de la regresión logística en 0,9825 (medida: 0,9883) y una
curva de aprendizaje con la regresión logística en 0,7251 al 5 % del entrenamiento (medida: 0,9298) — un cruce
inventado que confirmaba la tesis de Ng y Jordan. El reporte no decía que fueran estimaciones.**


#### Entregables:

La salida del literal se extrae en un archivo .txt dentro de la carpeta resultados `resultados/salida_0a.txt`, y es la siguiente:

`Medido ejecutando el código (sklearn, semilla 42):`
   `NB accuracy    0.9474`
   `NB F1 macro    0.9429`
   `LR accuracy    0.9883`
   `LR F1 macro    0.9875`

`LLM sin herramientas: zai-org/GLM-5.3-Flash, 2026-10-03, 13279 tokens de salida, 105.7 s, fin=stop`
   `reporte en parte0/reporte_sin_ejecutar.md; 29 cifras con decimales:`
   `37.3 %, 62.7 %, 0.9415, 0.9367, 0.9825, 0.9812, 0.9181, 0.9298, 0.9357, 0.9415, 0.9415, 0.8889, 0.9415, 0.9649, 0.9766, 0.9825, 0.8889, 0.9181, 0.9357, 99.4 %...` 



De esta salida se puede extraer que el id del modelo es `zai-org/GLM-5.3-Flash`, con fecha 2026-10-03, 12 028 tokens de salida, 61,72 s, fin=stop. La ruta es H200 de la USFQ por VPN (Global Protect).

1. **¿Por qué estar cerca no las hace aceptables?** Respuesta: La tarea A consiste en contrastar generativo vs discriminativo , por lo que se pide entrenar Naive Bayes (generativo) y regresión logística (discriminativo). Que el accuracy de NB y de LR estén cerca (0,9474 y 0,9883)  garantiza la exactitud de sus propios resultados vs los reales, y que el valor F1 de NB y LR estén de igual manera, cerca (0,9415 y 0,9825),  garantiza el rendimiento con pocos errores en general. Sin embargo, dichas métricas no salieron de una ejecución que no se puede reproducir ni auditar; el modelo las reconstruyó de memoria sobre un dataset propio de Python.

2. **¿Qué cifra habría cambiado una conclusión?** Respuesta: Dentro de `reporte_sin_ejecutar.md`, tanto en la tabla de la Parte 3 (n = 20: Naive Bayes 0.9123 y regresión logística 0.9064) como en la Discusión y las Conclusiones que se apoyan en ella, se verifica contra medido.json que esa curva no tiene ninguna ejecución detrás: el archivo solo trae la evaluación con el 100 % del entrenamiento, así que las celdas de n = 20, 40, 100 y 199 no se pueden comprobar. Por tanto, la cifra que habría cambiado una conclusión puede ser la de n = 20, porque el «cruce de Ng y Jordan» y la regla «con ≤ 20 ejemplos conviene Naive Bayes» descansan en una diferencia de un solo acierto sobre 171 pruebas.

3. **¿Qué debería comprobar un verificador?** Respuesta: Se debería comprobar que un verificador compruebe que cada cifra del reporte aparezca en un archivo escrito por una ejecución (el JSON de resultados de un script que terminó bien) o en el enunciado, y devolver por nombre las que no tengan origen, sin necesitar conocer las respuestas correctas.

### 0.b — El RAG plano que pierde la referencia

**python parte0/b_rag_plano.py
. Parte el enunciado de la Tarea A en un fragmento por sección, lo indexa con
TF-IDF y recupera los dos fragmentos más parecidos a la pregunta que el programador de la Parte 3 necesita
contestar: ¿con qué datos y con qué partición se calcula la curva de aprendizaje? Los dos que vuelven son la Parte 4
y la Parte 3, y ninguno dice cuáles son los datos: eso solo está escrito en la Parte 1, que no se parece a la pregunta.
Después sigue la arista 
depende_de que la mención «Parte 1» dentro de la Parte 3 crea en un grafo, y el contexto
pasa de 0 a 3 de los tres hechos.**

#### Entregables

La salida del literal se extrae en un archivo .txt dentro de la carpeta resultados `resultados/salida_0b.txt`, y es la siguiente:

`Pregunta del programador: «¿Con qué datos y con qué partición se calcula la curva de aprendizaje?»`

`1) RAG plano: los 2 fragmentos más similares (TF-IDF, coseno)`
   `Parte 4      sim=0.293`
   `Parte 3      sim=0.243`
   `El contexto cubre: ✗ conjunto de datos  ✗ proporción de prueba  ✗ prueba intocable`

`2) Los mismos 2 fragmentos, más un salto por las aristas depende_de del grafo`
   `Parte 4 —depende_de→ (ninguna)`
   `Parte 3 —depende_de→ Parte 1`
   `El contexto cubre: ✓ conjunto de datos  ✓ proporción de prueba  ✓ prueba intocable`

`Las aristas que el regex encontró en todo el enunciado:`
   `Parte 3 → Parte 1`
   `El reporte → Parte 2, Parte 3`

`RAG plano: 0 de 3 hechos. Con el salto: 3 de 3.`



1. **¿Por qué esta falla es la de recuperación y no de la generación?** Respuesta: Del archivo `tarea-a-generativo-discriminativo.pdf`, sobre el que se realizó el RAG, la Parte 1 consiste en cargar `load_breast_cancer` y dividir los datos 70 % / 30 % estratificando por la clase con `random_state=42`, dejando el conjunto de prueba intocable; es decir, ahí está escrita la respuesta a la pregunta del programador (con qué datos y con qué partición se calcula la curva de aprendizaje). Sin embargo, el RAG plano solo recuperó la Parte 4 (sim = 0.293) y la Parte 3 (sim = 0.243), y el contexto cubrió 0 de 3 hechos. Como el generador no puede usar lo que no recibió, y al agregar la Parte 1 por la arista `depende_de` el contexto sube a 3 de 3 sin cambiar nada del generador, la falla está en la recuperación y no en la generación.

2. **¿Qué haría un LLM con el primer contexto?** Respuesta: Con un contexto que no dice qué conjunto de datos ni qué proporción de prueba se usa, el LLM probablemente rellenaría el hueco con valores que él mismo supone y repitiendo tokens o frases que los presentaría con seguridad, sin avisar que le faltaba información.

3. **¿Por qué la arista se extrae con una regla y no se le pide al LLM?** Respuesta: El enunciado de la Tarea A se modela como un grafo: cada sección (Parte 1, Parte 2, etc.) es un nodo, y una arista depende_de es una flecha de una sección a otra que necesita para entenderse. La arista se extrae con una regla y no se le pide al LLM porque  una referencia cruzada como «la misma división de la Parte 1» es un patrón literal del texto que una expresión regular encuentra siempre, igual en cada corrida y sin gastar tokens: en el enunciado de la Tarea A se halló `Parte 3 → Parte 1` y `El reporte → Parte 2, Parte 3`. Un LLM, en cambio, puede omitir una arista o inventar otra, y dar un resultado distinto en cada ejecución. Esas aristas son lo que permite traer la Parte 1 al contexto de la Parte 3 (de 0 a 3 hechos), así que si el modelo se equivocara al extraerlas, volvería la misma falla sin que nada avisara. Por eso el esqueleto del grafo debe ser determinístico y verificable, y el LLM se reserva para lo que una regla no puede hacer, como extraer entidades y relaciones.

### 0.c —  Un returncode == 0 no es un experimento

**python parte0/c_exit_cero.py. Un «programador» guionizado entrega un árbol de decisión evaluado sobre los
mismos datos con que se entrenó: imprime una exactitud de 1,000, termina con código 0 y escribe su
resultados.json. El evaluador que solo mira el código de salida y el archivo lo aprueba. Dos comprobaciones de código —una estática sobre el árbol sintáctico y una de plausibilidad— lo rechazan.**

#### Entregables

La salida del literal se extrae en un archivo .txt dentro de la carpeta resultados `resultados/salida_0b.txt`, y es la siguiente:

`Pregunta del programador: «¿Con qué datos y con qué partición se calcula la curva de aprendizaje?»`

`1) RAG plano: los 2 fragmentos más similares (TF-IDF, coseno)`
   `Parte 4      sim=0.293`
   `Parte 3      sim=0.243`
   `El contexto cubre: ✗ conjunto de datos  ✗ proporción de prueba  ✗ prueba intocable`

`2) Los mismos 2 fragmentos, más un salto por las aristas depende_de del grafo`
   `Parte 4 —depende_de→ (ninguna)`
   `Parte 3 —depende_de→ Parte 1`
   `El contexto cubre: ✓ conjunto de datos  ✓ proporción de prueba  ✓ prueba intocable`

`Las aristas que el regex encontró en todo el enunciado:`
   `Parte 3 → Parte 1`
   `El reporte → Parte 2, Parte 3`

`RAG plano: 0 de 3 hechos. Con el salto: 3 de 3.`



1. **¿Qué otra fuga no detectaría la comprobación estática (piensa en un escalador ajustado con todos los datos)?** Respuesta: 
2. **¿Por qué la decisión final puede quedar en manos de un LLM crítico si las comprobaciones de código van antes.?** Respuesta: 